# AirShift — Deterioration Event Definition

This notebook defines what constitutes a future air quality deterioration event for the AirShift early warning system.

The goal is to establish a clear and measurable definition of deterioration that can later be used to create target labels for machine learning.

The definition focuses on:

* The pollutant used to represent deterioration
* The magnitude of the deterioration
* The future time window in which deterioration is evaluated

No target labels are created in this notebook. The defined event will be used in the following labeling stage.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
DATA_DIR = Path("../data/processed")

feature_file = DATA_DIR / "airshift_feature_engineered.csv"

df = pd.read_csv(feature_file, parse_dates=["datetime"])

print("Dataset shape:", df.shape)
print("Number of stations:", df["station"].nunique())
print("Date range:", df["datetime"].min(), "to", df["datetime"].max())

Dataset shape: (420768, 99)
Number of stations: 12
Date range: 2013-03-01 00:00:00 to 2017-02-28 23:00:00


## 1. Define the Deterioration Event

AirShift focuses on detecting significant future deterioration in air quality before it occurs.

For this project, **PM2.5** is used as the primary pollutant for defining a deterioration event because it is an important indicator of particulate air pollution.

A deterioration event is defined as a **30% or greater increase in PM2.5 within the following 6 hours** compared with the PM2.5 level at the prediction time.

The event definition is therefore based on:

* **Target pollutant:** PM2.5
* **Deterioration threshold:** 30% increase
* **Future horizon:** 6 hours

The future deterioration value will be used only to define the target in the labeling stage. It will not be used as an input feature for model prediction.


## 1. Define the Deterioration Event

AirShift focuses on detecting significant future deterioration in air quality before it occurs.

For this project, **PM2.5** is used as the primary pollutant for defining a deterioration event because it is an important indicator of particulate air pollution.

A deterioration event is defined as a **30% or greater increase in PM2.5 within the following 6 hours** compared with the PM2.5 level at the prediction time.

The event definition is therefore based on:

* **Target pollutant:** PM2.5
* **Deterioration threshold:** 30% increase
* **Future horizon:** 6 hours

The future deterioration value will be used only to define the target in the labeling stage. It will not be used as an input feature for model prediction.


In [3]:
TARGET_POLLUTANT = "PM2.5"

DETERIORATION_THRESHOLD = 0.30

FUTURE_HORIZON_HOURS = 6

print("Target pollutant:", TARGET_POLLUTANT)
print("Deterioration threshold:", f"{DETERIORATION_THRESHOLD * 100:.0f}%")
print("Future horizon:", f"{FUTURE_HORIZON_HOURS} hours")

Target pollutant: PM2.5
Deterioration threshold: 30%
Future horizon: 6 hours


## 2. Calculate Future PM2.5 Values

To define a future deterioration event, the PM2.5 concentration at the prediction time is compared with its value 6 hours later.

The future PM2.5 value is calculated separately for each monitoring station to preserve the temporal structure of the data.

These future values are used only to evaluate the deterioration event definition and will later be converted into target labels in the labeling stage.


In [4]:
df = df.sort_values(
    ["station", "datetime"]
).reset_index(drop=True)

df["PM2.5_future_6h"] = (
    df.groupby("station")["PM2.5"]
      .shift(-FUTURE_HORIZON_HOURS)
)

print("Future PM2.5 values calculated successfully.")

df[
    [
        "station",
        "datetime",
        "PM2.5",
        "PM2.5_future_6h"
    ]
].head(10)

Future PM2.5 values calculated successfully.


,station,datetime,PM2.5,PM2.5_future_6h
0,Aotizhongxin,2013-03-01 00:00:00,4.0,3.0
1,Aotizhongxin,2013-03-01 01:00:00,8.0,3.0
2,Aotizhongxin,2013-03-01 02:00:00,7.0,3.0
3,Aotizhongxin,2013-03-01 03:00:00,6.0,3.0
4,Aotizhongxin,2013-03-01 04:00:00,3.0,3.0
5,Aotizhongxin,2013-03-01 05:00:00,5.0,3.0
6,Aotizhongxin,2013-03-01 06:00:00,3.0,3.0
7,Aotizhongxin,2013-03-01 07:00:00,3.0,3.0
8,Aotizhongxin,2013-03-01 08:00:00,3.0,6.0
9,Aotizhongxin,2013-03-01 09:00:00,3.0,8.0


### Future PM2.5 Values Findings

The future PM2.5 values were calculated successfully for each monitoring station using a 6-hour horizon.

The `PM2.5_future_6h` feature represents the PM2.5 concentration measured six hours after each observation at the same monitoring station.

This future value will be used in the next step to calculate the relative change in PM2.5 and determine whether the 30% deterioration threshold is reached.

No target labels are created at this stage.


## 3. Calculate Future PM2.5 Change

To measure the magnitude of future air quality deterioration, the relative change in PM2.5 over the 6-hour future horizon is calculated.

The relative change is calculated by comparing the future PM2.5 concentration with the current PM2.5 concentration.

A positive value indicates an increase in PM2.5, while a negative value indicates a decrease.

The resulting feature will be used to evaluate whether the predefined 30% deterioration threshold is reached.


In [5]:
df["PM2.5_future_change_6h"] = np.where(
    df["PM2.5"] > 0,
    (df["PM2.5_future_6h"] - df["PM2.5"]) / df["PM2.5"],
    np.nan
)

print("Future PM2.5 change calculated successfully.")

df[
    [
        "station",
        "datetime",
        "PM2.5",
        "PM2.5_future_6h",
        "PM2.5_future_change_6h"
    ]
].head(10)

Future PM2.5 change calculated successfully.


,station,datetime,PM2.5,PM2.5_future_6h,PM2.5_future_change_6h
0,Aotizhongxin,2013-03-01 00:00:00,4.0,3.0,-0.250000
1,Aotizhongxin,2013-03-01 01:00:00,8.0,3.0,-0.625000
2,Aotizhongxin,2013-03-01 02:00:00,7.0,3.0,-0.571429
3,Aotizhongxin,2013-03-01 03:00:00,6.0,3.0,-0.500000
4,Aotizhongxin,2013-03-01 04:00:00,3.0,3.0,0.000000
5,Aotizhongxin,2013-03-01 05:00:00,5.0,3.0,-0.400000
6,Aotizhongxin,2013-03-01 06:00:00,3.0,3.0,0.000000
7,Aotizhongxin,2013-03-01 07:00:00,3.0,3.0,0.000000
8,Aotizhongxin,2013-03-01 08:00:00,3.0,6.0,1.000000
9,Aotizhongxin,2013-03-01 09:00:00,3.0,8.0,1.666667


### Future PM2.5 Change Findings

The relative change in PM2.5 over the following 6 hours was calculated successfully.

Positive values indicate an increase in PM2.5, while negative values indicate a decrease. A value of `0` indicates no change between the current and future PM2.5 concentrations.

For example, an increase from 3.0 to 6.0 results in a relative change of 1.0, corresponding to a 100% increase.

The resulting relative change will be used to evaluate the predefined 30% deterioration threshold before creating target labels in the labeling stage.


## 4. Evaluate Deterioration Thresholds

The initial 30% deterioration threshold is a design choice rather than a fixed scientific standard.

To determine whether this threshold is appropriate for the AirShift dataset, several alternative thresholds are evaluated.

The tested thresholds represent different levels of future PM2.5 increase:

* 10% increase
* 20% increase
* 30% increase
* 50% increase
* 75% increase

The purpose of this analysis is to examine how the choice of threshold affects the frequency of potential deterioration events.

No target labels are created at this stage. The results will be used to select an appropriate threshold for the final labeling stage.


In [6]:
thresholds = [0.10, 0.20, 0.30, 0.50, 0.75]

threshold_results = []

valid_change = df["PM2.5_future_change_6h"].notna()

total_valid = valid_change.sum()

for threshold in thresholds:
    
    events = (
        df.loc[valid_change, "PM2.5_future_change_6h"]
        >= threshold
    )
    
    event_count = events.sum()
    event_percentage = (event_count / total_valid) * 100
    
    threshold_results.append({
        "threshold": threshold,
        "event_count": event_count,
        "event_percentage": event_percentage
    })

threshold_results = pd.DataFrame(threshold_results)

threshold_results

,threshold,event_count,event_percentage
0,0.10,198649,47.468153
1,0.20,169990,40.619945
2,0.30,143946,34.396603
3,0.50,106487,25.445591
4,0.75,75218,17.973710


## 5. Evaluate Thresholds Across Future Horizons

The deterioration threshold should be evaluated together with the future prediction horizon because both choices affect how deterioration events are defined.

To examine their combined effect, several deterioration thresholds are evaluated across multiple future horizons.

The tested future horizons are:

* 1 hour
* 3 hours
* 6 hours
* 12 hours

For each horizon, the following relative PM2.5 increase thresholds are evaluated:

* 10%
* 20%
* 30%
* 50%
* 75%

This analysis examines how frequently potential deterioration events occur under different combinations of threshold and future horizon.

The results will help determine an appropriate event definition for the AirShift early warning task before creating the final target labels.


In [7]:
thresholds = [0.10, 0.20, 0.30, 0.50, 0.75]
horizons = [1, 3, 6, 12]

horizon_results = []

for horizon in horizons:
    
    future_column = f"PM2.5_future_{horizon}h"
    
    df[future_column] = (
        df.groupby("station")["PM2.5"]
          .shift(-horizon)
    )
    
    change_column = f"PM2.5_future_change_{horizon}h"
    
    df[change_column] = np.where(
        df["PM2.5"] > 0,
        (df[future_column] - df["PM2.5"]) / df["PM2.5"],
        np.nan
    )
    
    valid_change = df[change_column].notna()
    total_valid = valid_change.sum()
    
    for threshold in thresholds:
        
        event_count = (
            df.loc[valid_change, change_column] >= threshold
        ).sum()
        
        event_percentage = (
            event_count / total_valid
        ) * 100
        
        horizon_results.append({
            "horizon_hours": horizon,
            "threshold": threshold,
            "event_count": event_count,
            "event_percentage": event_percentage
        })

horizon_results = pd.DataFrame(horizon_results)

horizon_results

C:\Users\HP\AppData\Local\Temp\ipykernel_6584\1867996400.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[change_column] = np.where(
C:\Users\HP\AppData\Local\Temp\ipykernel_6584\1867996400.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[future_column] = (


,horizon_hours,threshold,event_count,event_percentage
0,1,0.10,126472,30.198230
1,1,0.20,78302,18.696485
2,1,0.30,52922,12.636400
3,1,0.50,30410,7.261119
4,1,0.75,17363,4.145834
5,3,0.10,178749,42.695607
6,3,0.20,137813,32.917721
7,3,0.30,106461,25.429048
8,3,0.50,69541,16.610416
9,3,0.75,43928,10.492549


### Threshold and Horizon Analysis Findings

The analysis showed that longer future horizons generally result in a higher percentage of deterioration events, while higher thresholds reduce event frequency.

A **30% increase within 6 hours** resulted in **34.40%** potential deterioration events, providing a reasonable balance between event frequency and the short-term early warning objective.

Therefore, **30% within 6 hours** was selected as the initial deterioration-event definition for AirShift.
